<a href="https://colab.research.google.com/github/nalinkai/Data-Science-Project-Lifecycle/blob/main/Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Data Science Life Cycle Coursework/Hotel-A-train.csv')


Handle Duplicate Records

In [ ]:
# Keep the first occurrence and drop others, keeping one row per Reservation-id
df = df.drop_duplicates(subset='Reservation-id', keep='first')

Standardize Categorical Variables

In [ ]:
# Apply to all object columns, or specifically to the problem ones
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.lower()

Convert Date Columns to Datetime

In [ ]:
date_cols = ['Expected_checkin', 'Expected_checkout', 'Booking_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

In [ ]:
df.head()

,Reservation-id,Gender,Age,Ethnicity,Educational_Level,Income,Country_region,Hotel_Type,Expected_checkin,Expected_checkout,...,Meal_Type,Visted_Previously,Previous_Cancellations,Deposit_type,Booking_channel,Required_Car_Parking,Reservation_Status,Use_Promotion,Discount_Rate,Room_Rate
0,39428300,f,40,latino,grad,<25k,north,city hotel,2015-07-01,2015-07-02,...,bb,no,no,no deposit,online,yes,check-out,yes,10,218
1,77491756,f,49,latino,mid-school,50k -- 100k,east,city hotel,2015-07-01,2015-07-02,...,bb,no,no,refundable,online,yes,check-out,no,0,185
2,73747291,f,42,caucasian,grad,<25k,east,city hotel,2015-07-02,2015-07-06,...,bb,no,no,no deposit,online,yes,check-out,no,0,119
3,67301739,m,25,african american,college,>100k,south,airport hotels,2015-07-02,2015-07-03,...,bb,no,no,refundable,agent,yes,check-out,yes,5,144
4,77222321,f,62,latino,high-school,25k --50k,east,resort,2015-07-03,2015-07-04,...,bb,no,no,no deposit,direct,no,check-out,yes,10,242


Outliers handling

In [ ]:
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(col, "→", len(outliers), "outliers")

Reservation-id → 0 outliers
Age → 0 outliers
Adults → 2286 outliers
Children → 0 outliers
Babies → 0 outliers
Discount_Rate → 0 outliers
Room_Rate → 0 outliers


In [ ]:
import pandas as pd

# Assuming your dataframe is called df (train / val / test)
# Make sure date columns are datetime type

df['Expected_checkin']  = pd.to_datetime(df['Expected_checkin'],  errors='coerce')
df['Expected_checkout'] = pd.to_datetime(df['Expected_checkout'], errors='coerce')
df['Booking_date']      = pd.to_datetime(df['Booking_date'],      errors='coerce')

# ─── Lead time in days ────────────────────────────────
df['lead_time_days'] = (df['Expected_checkin'] - df['Booking_date']).dt.days

# Optional: also create stay duration (very useful feature)
df['stay_duration_days'] = (df['Expected_checkout'] - df['Expected_checkin']).dt.days

In [ ]:
# How many negative values?
negative_count = (df['lead_time_days'] < 0).sum()

print(f"Number of rows with negative lead_time_days: {negative_count}")

if negative_count > 0:
    print(f"→ This is {negative_count / len(df) * 100:.2f}% of the data")
else:
    print("→ No negative values found. Good!")

Number of rows with negative lead_time_days: 506
→ This is 1.84% of the data


In [ ]:
# Before dropping - optional: see the shape
print("Shape before:", df.shape)

# Drop rows where lead_time_days is negative
df = df[df['lead_time_days'] >= 0].copy()   # .copy() avoids SettingWithCopyWarning

print("Shape after:", df.shape)
print("Rows removed:", 506)  # or calculate: original_rows - df.shape[0]

Shape before: (27495, 26)
Shape after: (26989, 26)
Rows removed: 506


In [ ]:
import pandas as pd
from google.colab import files

# Save file
df.to_csv("/content/drive/MyDrive/Data Science Life Cycle Coursework/Hotel-A-train_processed_data.csv", index=False)